In [ ]:
from pathlib import Path
import pandas as pd

folder = Path("../data/raw/system_data")

for f in sorted(folder.glob("*.xlsx")):
    xl = pd.ExcelFile(f)
    print("\n=====", f.name)
    print("sheets:", xl.sheet_names)
    df = xl.parse(xl.sheet_names[0])
    print("shape:", df.shape)
    print("columns:", list(df.columns))
    print("first:", df["DateTime"].min(), "| last:", df["DateTime"].max())

In [ ]:
for f in sorted(folder.glob("*.xlsx")):
    df = pd.read_excel(f, sheet_name="System Data",
                       usecols=["DateTime", "GMT Offset", "SNSP"])
    print("\n=====", f.name)
    print("SNSP min / mean / max:", df["SNSP"].min(), round(df["SNSP"].mean(), 3), df["SNSP"].max())
    print("missing SNSP:", df["SNSP"].isna().sum())
    print("duplicate DateTime + GMT Offset:", df.duplicated(["DateTime", "GMT Offset"]).sum())
    print("duplicate DateTime alone:", df.duplicated("DateTime").sum())
    print("GMT Offset values:", sorted(df["GMT Offset"].dropna().unique()))

In [ ]:
dd_folder = Path("../data/raw/dispatch_down")

for f in sorted(dd_folder.glob("DD-HH*.xlsx")):
    xl = pd.ExcelFile(f)
    print("\n=====", f.name)
    print("sheets:", xl.sheet_names)
    df = xl.parse("DD HH")
    print("shape:", df.shape)
    print("columns:", list(df.columns))
    print("first:", df["HH_TIMESTAMP"].min(), "| last:", df["HH_TIMESTAMP"].max())
    print(df[["UT_TYPE", "JURISDICTION"]].value_counts())

In [ ]:
dd25 = pd.read_excel(dd_folder / "DD-HH-2025-v10.xlsx", sheet_name="DD HH")
dd25.columns = [c.replace("Sum of ", "") for c in dd25.columns]
dd25["utc"] = dd25["HH_TIMESTAMP"] - pd.to_timedelta(dd25["GMT_OFFSET"], unit="h")

wind = dd25[dd25["UT_TYPE"] == "Wind"]
print("duplicate utc per jurisdiction:", wind.duplicated(["JURISDICTION", "utc"]).sum())

ie = wind[wind["JURISDICTION"] == "IE"]
mask = (ie["HH_TIMESTAMP"] >= "2025-10-26 00:30") & (ie["HH_TIMESTAMP"] <= "2025-10-26 02:30")
print(ie.loc[mask, ["HH_TIMESTAMP", "GMT_OFFSET", "utc"]])

In [ ]:
DD_COLUMNS = ["UT_TYPE", "JURISDICTION", "HH_TIMESTAMP", "GMT_OFFSET",
              "AV_MWH", "AO_MWH", "HI_FRQ_MIN_GEN_MWH", "ROCOF_INERTIA_MWH",
              "SNSP_MWH", "TRANS_CONSTR_MWH", "TSO_TEST_MWH",
              "DD_MWH", "CURTAILMENTS_MWH", "CONSTRAINTS_MWH", "OTHER_MWH"]

def load_dd_file(path):
    df = pd.read_excel(path, sheet_name="DD HH")
    df.columns = [c.replace("Sum of ", "") for c in df.columns]
    df = df[DD_COLUMNS].copy()
    df["utc"] = df["HH_TIMESTAMP"] - pd.to_timedelta(df["GMT_OFFSET"], unit="h")
    df["source_file"] = Path(path).name
    return df.sort_values(["UT_TYPE", "JURISDICTION", "utc"]).reset_index(drop=True)

dd25 = load_dd_file(dd_folder / "DD-HH-2025-v10.xlsx")
print(dd25.shape)
print("repeated utc:", dd25.duplicated(["UT_TYPE", "JURISDICTION", "utc"]).sum())
steps = dd25.groupby(["UT_TYPE", "JURISDICTION"])["utc"].diff().dropna()
print(steps.value_counts())

In [ ]:
dd_files = sorted(dd_folder.glob("DD-HH*.xlsx"))
dd_all = pd.concat([load_dd_file(f) for f in dd_files], ignore_index=True)
dd_all = dd_all.sort_values(["UT_TYPE", "JURISDICTION", "utc"]).reset_index(drop=True)

print(dd_all.shape)
print(dd_all.groupby(["UT_TYPE", "JURISDICTION"]).size())
print("repeated utc:", dd_all.duplicated(["UT_TYPE", "JURISDICTION", "utc"]).sum())
steps = dd_all.groupby(["UT_TYPE", "JURISDICTION"])["utc"].diff().dropna()
print(steps.value_counts())

In [ ]:
c = dd_all

resid_dd  = (c["DD_MWH"] - (c["CONSTRAINTS_MWH"] + c["CURTAILMENTS_MWH"])).abs()
resid_con = (c["CONSTRAINTS_MWH"] - (c["TRANS_CONSTR_MWH"] + c["TSO_TEST_MWH"])).abs()
resid_cur = (c["CURTAILMENTS_MWH"] - (c["HI_FRQ_MIN_GEN_MWH"] + c["ROCOF_INERTIA_MWH"] + c["SNSP_MWH"])).abs()
print("max residuals:", round(resid_dd.max(), 3), round(resid_con.max(), 3), round(resid_cur.max(), 3))

print("nulls:", int(c[DD_COLUMNS].isna().sum().sum()))
print("negatives:", int((c[["AV_MWH", "AO_MWH", "DD_MWH", "CURTAILMENTS_MWH", "CONSTRAINTS_MWH"]] < 0).sum().sum()))
print("output above availability:", int((c["AO_MWH"] > c["AV_MWH"] + 1e-6).sum()))

c["year"] = c["HH_TIMESTAMP"].dt.year
w = c[c["UT_TYPE"] == "Wind"]
yr = w.groupby("year")[["AV_MWH", "DD_MWH", "CONSTRAINTS_MWH", "CURTAILMENTS_MWH"]].sum()
for col in ["DD_MWH", "CONSTRAINTS_MWH", "CURTAILMENTS_MWH"]:
    yr[col.replace("_MWH", "_pct")] = (yr[col] / yr["AV_MWH"] * 100).round(1)
print(yr[["DD_pct", "CONSTRAINTS_pct", "CURTAILMENTS_pct"]])

In [ ]:
SD_COLUMNS = ["DateTime", "GMT Offset", "IE Demand", "NI Demand",
              "IE Wind Availability", "NI Wind Availability",
              "IE Wind Generation", "NI Wind Generation", "SNSP"]

def load_sd_file(path):
    df = pd.read_excel(path, sheet_name="System Data", usecols=SD_COLUMNS)
    df = df.rename(columns={"DateTime": "local_time", "GMT Offset": "gmt_offset"})
    df["utc"] = df["local_time"] - pd.to_timedelta(df["gmt_offset"], unit="h")
    df["source_file"] = Path(path).name
    return df.sort_values("utc").reset_index(drop=True)

sd_files = sorted(folder.glob("*.xlsx"))
sd_all = pd.concat([load_sd_file(f) for f in sd_files], ignore_index=True)
sd_all = sd_all.sort_values("utc").reset_index(drop=True)

print(sd_all.shape)
print("repeated utc:", sd_all["utc"].duplicated().sum())
print("nulls:", int(sd_all[SD_COLUMNS[2:]].isna().sum().sum()))
print(sd_all["utc"].diff().dropna().value_counts())
print(sd_all[SD_COLUMNS[2:]].describe().loc[["min", "max"]].round(3))